# SINGLE CELL DIFFERENTIAL GENE EXPRESSION BURDEN SCORE

In [ ]:
# Load packages 
import re
import os
import gc
import glob
import pandas as pd 
from biomart import BiomartServer
from functools import reduce

# Set bucket environment
my_bucket = os.getenv('WORKSPACE_BUCKET')

In [ ]:
# Remove previously generated polygenic risk score (PRS) files
!rm -rf *PRS.tsv

# Remove any existing gene list directory and recreate it
!rm -rf gene_list
!mkdir -p gene_list 

# Download the reference gene sets from the workspace bucket
# These files contain the gene identifiers used to define each gene set
!gsutil cp {my_bucket}/data/gene_burden_analysis/gene_sets/single_cell_differential_expression/* gene_list

# Create a directory to store the aggregated gene burden scores
!mkdir -p burden_sum_scores

In [ ]:
%%bash 
# Set workspace bucket containing the gene burden sum scores 
my_bucket="gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe"

# Create directory for gene burden sum scores 
mkdir -p burden_scores 

# Loop to download file for the burden sum score for each gene 
for chrom in {1..22}; do

  # Create a separate directory for each chromosome
  mkdir -p burden_scores/burden_scores_chr${chrom}
  
  # Download all gene burden sum score files for the current chromosome
  # Files are copied into their corresponding chromosome-specific directory
  gsutil cp "${my_bucket}/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr${chrom}/*.sscore" burden_scores/burden_scores_chr${chrom}
  
done

In [ ]:
# Define the directory containing the reference gene sets
gene_folder = "gene_list"
# Define the directory containing the reference gene sets
gene_files = sorted(glob.glob(os.path.join(gene_folder, "*")))

# Iterate over each reference gene set
for gene_filename in gene_files: 
    
    # Load the reference gene set containing the Ensembl gene identifiers
    genes = pd.read_csv(gene_filename, sep='\t')
    
    # Remove Ensembl version numbers to ensure compatibility with the gene identifiers
    # used in the chromosome-specific burden score files
    genes["gene_id"] = genes['ensembl_gene_id'].str.split('.').str[0]
    
    # Extract the gene set name from the input filename
    # This name will be used to identify the resulting PRS file
    match = re.search(r'/([^/]+)\.tsv$', gene_filename)
    if match:
        name = match.group(1)
    
    # Initialize the master dataframe outside the chromosome loop
    # This dataframe will accumulate the burden scores for all genes across all chromosomes
    master = None 
    
    # Iterate over autosomes 1–22 to aggregate gene burden scores
    for chrom in range(1, 23):
        
        print(f"Processing chromosome {chrom} ....")

        # Define the directory containing the burden score files for the current chromosome
        folder_path = f"burden_scores/burden_scores_chr{chrom}"
        
        # Define the directory containing the burden score files for the current chromosome
        files = sorted(glob.glob(os.path.join(folder_path, "*")))

        # Skip the current chromosome if no burden score files are available
        if not files:
            print(f"No files found for chromosome {chrom}, skipping...")
            continue
            
        # Initialize the master dataframe using the first available burden score file
        # Retain participant identifiers and initialize the cumulative gene burden score to zero
        if master is None:
            # If first chromosome, assign master as merged file 
            master = pd.read_csv(files[0], sep="\t")[["#FID", "IID"]]
            
            # Initialize the master dataframe using the first available burden score file
            # Retain participant identifiers and initialize the cumulative gene burden score to zero
            master["TOTAL_SCORE1_SUM"] = 0.0 
        else:
            # Retain participant identifiers and initialize the cumulative gene burden score to zero
            master = master

        # Iterate over the gene-specific burden score files for the current chromosome
        for filename in files:
            
            # Extract the Ensembl gene identifier from the filename
            match = re.search(r"(ENSG\d+)", filename)
            
            # Skip files that do not contain a recognizable Ensembl gene identifier
            if not match:
                continue
            ensg_id = match.group(1)
            
            # Check whether the gene belongs to the current reference gene set
            # Skip genes that are not included in the gene set
            if ensg_id not in genes["gene_id"].unique():
                continue
                
            # Report the gene currently being processed
            print(f"Processing gene {ensg_id} ....")

            # Load the gene-specific burden score file
            df = pd.read_csv(filename, sep="\t")
            
            # Verify that the sum of named allele dosages matches the reported gene burden score
            # Skip files that fail this consistency check
            if not (df["NAMED_ALLELE_DOSAGE_SUM"] == df["SCORE1_SUM"]).all():
                continue

            # Merge the current gene's burden scores into the master dataframe
            # Participants are matched using their family and individual identifiers
            # An outer join retains participants even if they are absent from one of the files
            master = master.merge(df[["#FID", "IID", "SCORE1_SUM"]],
                                  on=["#FID", "IID"], how="outer")

            # Add the current gene's burden score to the cumulative gene-set score
            master["TOTAL_SCORE1_SUM"] += master["SCORE1_SUM"]
            
            # Remove the individual gene score column after incorporating it into the cumulative score
            master.drop(columns=["SCORE1_SUM"], inplace=True)
            
        # Report the dimensions of the master dataframe after processing the current chromosome
        print(f"Chromosome {chrom} merge complete. Master shape: {master.shape}")
            
    # Report completion of gene burden score aggregation across all chromosomes
    print("All chromosomes merged.")
    print("Final shape:", master.shape)
    
    # Export the aggregated gene-set burden scores as a tab-delimited file
    # Each row represents a participant and TOTAL_SCORE1_SUM represents their cumulative score
    master.to_csv(f"{name}-PRS.tsv", sep ='\t', index=False)


In [37]:
# Save prs files to workspace bucket 
!gsutil cp *PRS.tsv {my_bucket}/data/gene_burden_analysis/burden_gene_sets/single_cell_differential_expression/ 